In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.io import fits
from astropy.table import Table, vstack
from astroquery.mast import Observations

C_CGS = 2.99792458e10  # cm/s; used for Jy -> f_lambda
DATA_DIR = Path("data")
FITS_DIR = DATA_DIR / "miri_x1d"  # MAST Level-3 *_x1d.fits
SPEC_DIR = DATA_DIR / "miri_spectra_csv"  # pipeline-ready Wavelength/Flux CSVs

## 1. Get MIRI spectroscopy in MAST

In [2]:
obs = Observations.query_criteria(
    obs_collection="JWST",
    instrument_name=["MIRI/SLIT", "MIRI/SLITLESS", "MIRI/IFU"],
)
miri = SkyCoord(obs["s_ra"], obs["s_dec"], unit="deg")

## 2. Load DJA NIRSpec sources

In [3]:
dja = Table.read("data/dja_nirspec_z>9_grade3.csv")
dja_coords = SkyCoord(dja["ra"], dja["dec"], unit="deg")
print(f"{len(dja)} DJA sources, {len(obs)} MIRI observations")

177 DJA sources, 41230 MIRI observations


## 3. Match within 1 arcsec

In [4]:
# Nearest MIRI pointing per DJA row; keep matches within 1 arcsec
idx, sep2d, _ = dja_coords.match_to_catalog_sky(miri)
matched = sep2d < 1 * u.arcsec

hits = dja[matched].copy()
hits["sep_arcsec"] = sep2d[matched].to(u.arcsec).value
for col in ["obsid", "proposal_id", "instrument_name", "obs_id", "s_ra", "s_dec", "t_exptime"]:
    if col in obs.colnames:
        hits[f"miri_{col}"] = obs[col][idx[matched]]

hits.sort("sep_arcsec")

# Collapse duplicate DJA entries at the same sky position; keep closest MIRI match
unique = hits.copy()
unique["ra_round"] = np.round(unique["ra"].astype(float), 6)
unique["dec_round"] = np.round(unique["dec"].astype(float), 6)
_, first = np.unique(
    np.array([unique["ra_round"], unique["dec_round"]]).T,
    axis=0,
    return_index=True,
)
unique = unique[np.sort(first)]

print(f"{matched.sum()} / {len(dja)} DJA rows within 1\" ({len(unique)} unique positions)")
out_cols = [
    "root", "srcid", "ra", "dec", "z", "grade", "sep_arcsec",
    "miri_obs_id", "miri_instrument_name", "miri_proposal_id",
]
out_cols = [c for c in out_cols if c in unique.colnames]
unique[out_cols].write("data/miri_dja_matches_1arcsec.csv", overwrite=True)
print("wrote data/miri_dja_matches_1arcsec.csv")
unique[out_cols]

55 / 177 DJA rows within 1" (24 unique positions)
wrote data/miri_dja_matches_1arcsec.csv


root,srcid,ra,dec,z,grade,sep_arcsec,miri_obs_id,miri_instrument_name,miri_proposal_id
str24,int64,float64,float64,float64,int64,float64,str44,str13,str5
jades-gdn09-v4,55757,189.2176869,62.199492,9.7419,3,0.0012388515955346353,jw08051-o018_t009_miri_p750l,MIRI/SLIT,8051
uncover-v4,13151,3.59250134,-30.40146429,9.8015,3,0.0014707917629555494,jw08051-o015_t007_miri_p750l,MIRI/SLIT,8051
uncover-v4,26185,3.5670708,-30.37786065,10.0628,3,0.0016632249474689566,jw08051-o012_t005_miri_p750l,MIRI/SLIT,8051
abell2744-castellano2-v4,10896,3.5119292,-30.3718588,9.8817,3,0.00168318719713927,jw08051-o017_t008_miri_p750l,MIRI/SLIT,8051
uncover-v4,37126,3.59011077,-30.35974219,10.3781,3,0.0018942440545210869,jw08051-o006_t002_miri_p750l,MIRI/SLIT,8051
jades-gds1-v4,20088041,53.1755177,-27.7806356,9.7127,3,0.0019471476529077656,jw08051-o021_t010_miri_p750l,MIRI/SLIT,8051
abell2744-castellano1-v4,22302,3.4513686,-30.320717,10.3955,3,0.002130953143498191,jw08051002001_xx105_00002_miri,MIRI/SLIT,8051
abell2744-castellano1-v4,23984,3.451428,-30.3217981,10.2471,3,0.0031705930603061817,jw08051-o008_t003_miri_p750l,MIRI/SLIT,8051
abell2744-castellano2-v4,17724,3.4787606,-30.3455157,10.1448,3,0.003554701327492141,jw08051-o009_t004_miri_p750l,MIRI/SLIT,8051


## 4. Download MIRI Level-3 `x1d` FITS

For each unique match, pull the MAST science `X1D` product at `calib_level=3` (combined 1D spectrum).
Some MAST rows are visit placeholders (`*_xx*` in `obs_id`) with no products; for those we fall back to the nearest association-level MIRI pointing that has an L3 `x1d`.

Programs still under exclusive access return HTTP 401; those matches are logged as `proprietary` and skipped in the CSV step.


In [7]:
def find_l3_x1d_products(obs_id):
    """Return MAST product table for L3 SCIENCE X1D FITS for one obs_id (or empty)."""
    obs_rows = Observations.query_criteria(obs_collection="JWST", obs_id=obs_id)
    if len(obs_rows) == 0:
        return obs_rows
    products = Observations.get_product_list(obs_rows)
    return Observations.filter_products(
        products,
        productType="SCIENCE",
        productSubGroupDescription="X1D",
        calib_level=[3],
        extension="fits",
    )


def resolve_download_obs_id(matched_obs_id, ra_deg, dec_deg, search_radius="30s"):
    """Prefer the matched obs_id; if it has no L3 X1D, use nearest association with one."""
    products = find_l3_x1d_products(matched_obs_id)
    if len(products) > 0:
        return matched_obs_id, products, "matched"

    # Placeholder / incomplete MAST rows: search nearby MIRI spectroscopy
    nearby = Observations.query_criteria(
        obs_collection="JWST",
        instrument_name=["MIRI/SLIT", "MIRI/SLITLESS", "MIRI/IFU"],
        coordinates=f"{ra_deg} {dec_deg}",
        radius=search_radius,
    )
    if len(nearby) == 0:
        return matched_obs_id, products, "missing"

    # Association products look like jwPPPPP-oOOO_tTTT_miri_...
    is_assoc = np.array([("-o" in oid) and ("xx" not in oid) for oid in nearby["obs_id"]])
    candidates = nearby[is_assoc] if is_assoc.any() else nearby
    target = SkyCoord(ra_deg, dec_deg, unit="deg")
    cand_coords = SkyCoord(candidates["s_ra"], candidates["s_dec"], unit="deg")
    order = np.argsort(target.separation(cand_coords))

    for index in order:
        candidate_obs_id = candidates["obs_id"][index]
        candidate_products = find_l3_x1d_products(candidate_obs_id)
        if len(candidate_products) > 0:
            return candidate_obs_id, candidate_products, "fallback"

    return matched_obs_id, products, "missing"

In [8]:
matches = Table.read(DATA_DIR / "miri_dja_matches_1arcsec.csv")
FITS_DIR.mkdir(parents=True, exist_ok=True)

# One download per unique matched MIRI obs_id
unique_obs = matches[np.unique(matches["miri_obs_id"], return_index=True)[1]]
download_rows = []

for row in unique_obs:
    matched_obs_id = str(row["miri_obs_id"])
    download_obs_id, products, status = resolve_download_obs_id(
        matched_obs_id, float(row["ra"]), float(row["dec"])
    )
    local_path = ""
    product_name = ""
    if status != "missing" and len(products) > 0:
        # Prefer the combined product named like <obs_id>_x1d.fits
        preferred = [
            product for product in products
            if str(product["productFilename"]) == f"{download_obs_id}_x1d.fits"
        ]
        if preferred:
            chosen_table = products[products["productFilename"] == f"{download_obs_id}_x1d.fits"]
        else:
            chosen_table = products[[0]]
        product_name = str(chosen_table["productFilename"][0])
        manifest = Observations.download_products(
            chosen_table,
            download_dir=str(FITS_DIR),
            mrp_only=False,
        )
        local_path = str(manifest["Local Path"][0])
        download_status = str(manifest["Status"][0])
        # Message column is present on failed MAST downloads (e.g. HTTP 401)
        message = str(manifest["Message"][0]) if "Message" in manifest.colnames else ""
        if download_status != "COMPLETE" or not Path(local_path).is_file():
            status = "proprietary" if "401" in message else "download_failed"
            local_path = ""
            print(f"{matched_obs_id}: {status} ({download_status}) {message}")
        else:
            print(f"{matched_obs_id}: {status} -> {product_name}")
    else:
        print(f"{matched_obs_id}: {status} -> NONE")

    download_rows.append({
        "miri_obs_id_matched": matched_obs_id,
        "miri_obs_id_download": download_obs_id,
        "resolve_status": status,
        "productFilename": product_name,
        "local_fits": local_path,
    })

download_log = Table(rows=download_rows)
download_log.write(DATA_DIR / "miri_x1d_download_log.csv", overwrite=True)
n_ok = sum(1 for record in download_rows if record["local_fits"])
print(f"downloaded {n_ok} / {len(download_rows)} unique MIRI obs")
download_log


INFO: Found cached file data/miri_x1d/mastDownload/JWST/jw01262-o005_t014_miri_ch3-long/jw01262-o005_t014_miri_ch3-long_x1d.fits with expected size 348480. [astroquery.query]
jw01262-o005_t014_miri_ch3-long: matched -> jw01262-o005_t014_miri_ch3-long_x1d.fits
INFO: Found cached file data/miri_x1d/mastDownload/JWST/jw02926-o002_t001_miri_ch2-medium/jw02926-o002_t001_miri_ch2-medium_x1d.fits with expected size 400320. [astroquery.query]
jw02926-o002_t001_miri_ch2-medium: matched -> jw02926-o002_t001_miri_ch2-medium_x1d.fits
INFO: Found cached file data/miri_x1d/mastDownload/JWST/jw03703-o002_t001_miri_p750l/jw03703-o002_t001_miri_p750l_x1d.fits with expected size 195840. [astroquery.query]
jw03703-o002_t001_miri_p750l: matched -> jw03703-o002_t001_miri_p750l_x1d.fits
INFO: Found cached file data/miri_x1d/mastDownload/JWST/jw03703-o006_t002_miri_p750l/jw03703-o006_t002_miri_p750l_x1d.fits with expected size 178560. [astroquery.query]
jw03703-o006_t002_miri_p750l: matched -> jw03703-o006_t

INFO: Found cached file data/miri_x1d/mastDownload/JWST/jw08051-o008_t003_miri_p750l/jw08051-o008_t003_miri_p750l_x1d.fits with expected size 198720. [astroquery.query]
jw08051002001_xx105_00002_miri: fallback -> jw08051-o008_t003_miri_p750l_x1d.fits
INFO: Found cached file data/miri_x1d/mastDownload/JWST/jw08544-o003_t001_miri_p750l/jw08544-o003_t001_miri_p750l_x1d.fits with expected size 195840. [astroquery.query]
jw08544-o003_t001_miri_p750l: matched -> jw08544-o003_t001_miri_p750l_x1d.fits
jw09165-o016_t005_miri_p750l: proprietary (ERROR) HTTPError: 401 Client Error: Unauthorized for url: https://mast.stsci.edu/api/v0.1/Download/file?uri=mast:JWST/product/jw09165-o016_t005_miri_p750l_x1d.fits
jw09425-o002_t001_miri_p750l: proprietary (ERROR) HTTPError: 401 Client Error: Unauthorized for url: https://mast.stsci.edu/api/v0.1/Download/file?uri=mast:JWST/product/jw09425-o002_t001_miri_p750l_x1d.fits
downloaded 17 / 19 unique MIRI obs


miri_obs_id_matched,miri_obs_id_download,resolve_status,productFilename,local_fits
str33,str33,str11,str42,str108
jw01262-o005_t014_miri_ch3-long,jw01262-o005_t014_miri_ch3-long,matched,jw01262-o005_t014_miri_ch3-long_x1d.fits,data/miri_x1d/mastDownload/JWST/jw01262-o005_t014_miri_ch3-long/jw01262-o005_t014_miri_ch3-long_x1d.fits
jw02926-o002_t001_miri_ch2-medium,jw02926-o002_t001_miri_ch2-medium,matched,jw02926-o002_t001_miri_ch2-medium_x1d.fits,data/miri_x1d/mastDownload/JWST/jw02926-o002_t001_miri_ch2-medium/jw02926-o002_t001_miri_ch2-medium_x1d.fits
jw03703-o002_t001_miri_p750l,jw03703-o002_t001_miri_p750l,matched,jw03703-o002_t001_miri_p750l_x1d.fits,data/miri_x1d/mastDownload/JWST/jw03703-o002_t001_miri_p750l/jw03703-o002_t001_miri_p750l_x1d.fits
jw03703-o006_t002_miri_p750l,jw03703-o006_t002_miri_p750l,matched,jw03703-o006_t002_miri_p750l_x1d.fits,data/miri_x1d/mastDownload/JWST/jw03703-o006_t002_miri_p750l/jw03703-o006_t002_miri_p750l_x1d.fits
jw04530-o001_t001_miri_ch4-long,jw04530-o001_t001_miri_ch4-long,matched,jw04530-o001_t001_miri_ch4-long_x1d.fits,data/miri_x1d/mastDownload/JWST/jw04530-o001_t001_miri_ch4-long/jw04530-o001_t001_miri_ch4-long_x1d.fits
jw04586-o002_t001_miri_ch2-medium,jw04586-o002_t001_miri_ch2-medium,matched,jw04586-o002_t001_miri_ch2-medium_x1d.fits,data/miri_x1d/mastDownload/JWST/jw04586-o002_t001_miri_ch2-medium/jw04586-o002_t001_miri_ch2-medium_x1d.fits
jw08051-o006_t002_miri_p750l,jw08051-o006_t002_miri_p750l,matched,jw08051-o006_t002_miri_p750l_x1d.fits,data/miri_x1d/mastDownload/JWST/jw08051-o006_t002_miri_p750l/jw08051-o006_t002_miri_p750l_x1d.fits
jw08051-o008_t003_miri_p750l,jw08051-o008_t003_miri_p750l,matched,jw08051-o008_t003_miri_p750l_x1d.fits,data/miri_x1d/mastDownload/JWST/jw08051-o008_t003_miri_p750l/jw08051-o008_t003_miri_p750l_x1d.fits
jw08051-o009_t004_miri_p750l,jw08051-o009_t004_miri_p750l,matched,jw08051-o009_t004_miri_p750l_x1d.fits,data/miri_x1d/mastDownload/JWST/jw08051-o009_t004_miri_p750l/jw08051-o009_t004_miri_p750l_x1d.fits


## 5. Convert `x1d` FITS -> pipeline CSVs

`01_spectral_fitting.ipynb` expects CSVs with columns `Wavelength` (µm) and ` Flux` (f_λ in erg s⁻¹ cm⁻² Å⁻¹, already ×1e21).
MIRI L3 `x1d` stores wavelength in µm and flux in Jy; convert with the same Jy↔f_λ helper used in the fitting notebook.

In [9]:
def jy_to_flambda_1e21(flux_jy, wavelength_um):
    """Jy -> f_lambda [erg/s/cm^2/A] * 1e21, wavelength in microns."""
    f_nu = np.asarray(flux_jy, float) * 1e-23  # Jy -> erg/s/cm^2/Hz
    lam_cm = np.asarray(wavelength_um, float) * 1e-4  # um -> cm
    f_per_cm = f_nu * (C_CGS / lam_cm ** 2)  # f_lambda per cm
    f_per_A = f_per_cm * 1e-8  # per cm -> per Angstrom
    return f_per_A * 1e21


def x1d_to_arrays(fits_path):
    """Read EXTRACT1D WAVELENGTH (um), FLUX and FLUX_ERROR (Jy); drop non-finite flux."""
    with fits.open(fits_path) as hdul:
        table = hdul["EXTRACT1D"].data
        wavelength_um = np.asarray(table["WAVELENGTH"], float)
        flux_jy = np.asarray(table["FLUX"], float)
        err_jy = np.asarray(table["FLUX_ERROR"], float)
    good = np.isfinite(wavelength_um) & np.isfinite(flux_jy)
    return wavelength_um[good], flux_jy[good], err_jy[good]


SPEC_DIR.mkdir(parents=True, exist_ok=True)
download_log = Table.read(DATA_DIR / "miri_x1d_download_log.csv")
fits_by_matched = {
    str(row["miri_obs_id_matched"]): row
    for row in download_log
    if str(row["local_fits"]) and Path(str(row["local_fits"])).is_file()
}
status_by_matched = {
    str(row["miri_obs_id_matched"]): str(row["resolve_status"])
    for row in download_log
}

prepared_rows = []
for source in matches:
    matched_obs_id = str(source["miri_obs_id"])
    log_row = fits_by_matched.get(matched_obs_id)
    source_tag = f"{source['root']}_{source['srcid']}"
    csv_name = f"{source_tag}_miri_x1d.csv"
    csv_path = SPEC_DIR / csv_name
    status = status_by_matched.get(matched_obs_id, "no_fits")
    if status in ("matched", "fallback"):
        status = "no_fits"
    n_pix = 0
    download_obs_id = ""
    local_fits = ""

    if log_row is not None:
        local_fits = str(log_row["local_fits"])
        download_obs_id = str(log_row["miri_obs_id_download"])
        wavelength_um, flux_jy, err_jy = x1d_to_arrays(local_fits)
        flux_1e21 = jy_to_flambda_1e21(flux_jy, wavelength_um)
        # Same linear Jy->f_lambda transform for uncertainties
        err_1e21 = jy_to_flambda_1e21(err_jy, wavelength_um)
        err_1e21 = np.where(np.isfinite(err_1e21) & (err_1e21 > 0), err_1e21, np.nan)
        # Leading space in " Flux" matches JZ14_spec.csv / load_spectrum_csv
        frame = pd.DataFrame({
            "Wavelength": wavelength_um,
            " Flux": flux_1e21,
            "Flux_Error": err_1e21,
        })
        frame.to_csv(csv_path, index=False)
        status = "ok"
        n_pix = len(frame)

    prepared_rows.append({
        "root": source["root"],
        "srcid": int(source["srcid"]),
        "ra": float(source["ra"]),
        "dec": float(source["dec"]),
        "z": float(source["z"]),
        "grade": int(source["grade"]),
        "sep_arcsec": float(source["sep_arcsec"]),
        "miri_obs_id_matched": matched_obs_id,
        "miri_obs_id_download": download_obs_id,
        "miri_instrument_name": str(source["miri_instrument_name"]),
        "local_fits": local_fits,
        "spectrum_csv": str(csv_path) if status == "ok" else "",
        "n_pix": n_pix,
        "status": status,
    })
    print(f"{source_tag}: {status} n_pix={n_pix}")

prepared = Table(rows=prepared_rows)
prepared.write(DATA_DIR / "miri_dja_spectra_ready.csv", overwrite=True)
print(f"ready {sum(prepared['status'] == 'ok')} / {len(prepared)} sources -> {SPEC_DIR}")
prepared["root", "srcid", "z", "miri_obs_id_download", "spectrum_csv", "n_pix", "status"]


jades-gdn09-v4_55757: ok n_pix=387
uncover-v4_13151: ok n_pix=387
uncover-v4_26185: ok n_pix=387
abell2744-castellano2-v4_10896: ok n_pix=387
uncover-v4_37126: ok n_pix=387
jades-gds1-v4_20088041: ok n_pix=387
abell2744-castellano1-v4_22302: ok n_pix=387
abell2744-castellano1-v4_23984: ok n_pix=387
abell2744-castellano2-v4_17724: ok n_pix=387
jades-gds10-v4_183348: ok n_pix=389
abell2744-castellano2-v4_22600: ok n_pix=386
ceers-ddt-v4_1: ok n_pix=386
ceers-ddt-v4_10: ok n_pix=1124
capers-egs61-v4_22637: proprietary n_pix=0
uncover-v4_3686: ok n_pix=717
ceers-v4_80026: ok n_pix=387
goodsn-wide0-v4_1268: ok n_pix=1124
macs1149-stiavelli-v4_3: ok n_pix=1028
jades-gdn11-v4_5591: ok n_pix=1124
macs1149-v4_5110240: ok n_pix=1028
capers-udsp3-v4_126973: proprietary n_pix=0
jades-gds-wide2-v4_749: ok n_pix=387
jades-gds-wide2-v4_596: ok n_pix=387
jades-gds-wide2-v4_68: ok n_pix=387
ready 22 / 24 sources -> data/miri_spectra_csv


root,srcid,z,miri_obs_id_download,spectrum_csv,n_pix,status
str24,int64,float64,str33,str65,int64,str11
jades-gdn09-v4,55757,9.7419,jw08051-o018_t009_miri_p750l,data/miri_spectra_csv/jades-gdn09-v4_55757_miri_x1d.csv,387,ok
uncover-v4,13151,9.8015,jw08051-o015_t007_miri_p750l,data/miri_spectra_csv/uncover-v4_13151_miri_x1d.csv,387,ok
uncover-v4,26185,10.0628,jw08051-o012_t005_miri_p750l,data/miri_spectra_csv/uncover-v4_26185_miri_x1d.csv,387,ok
abell2744-castellano2-v4,10896,9.8817,jw08051-o017_t008_miri_p750l,data/miri_spectra_csv/abell2744-castellano2-v4_10896_miri_x1d.csv,387,ok
uncover-v4,37126,10.3781,jw08051-o006_t002_miri_p750l,data/miri_spectra_csv/uncover-v4_37126_miri_x1d.csv,387,ok
jades-gds1-v4,20088041,9.7127,jw08051-o021_t010_miri_p750l,data/miri_spectra_csv/jades-gds1-v4_20088041_miri_x1d.csv,387,ok
abell2744-castellano1-v4,22302,10.3955,jw08051-o008_t003_miri_p750l,data/miri_spectra_csv/abell2744-castellano1-v4_22302_miri_x1d.csv,387,ok
abell2744-castellano1-v4,23984,10.2471,jw08051-o008_t003_miri_p750l,data/miri_spectra_csv/abell2744-castellano1-v4_23984_miri_x1d.csv,387,ok
abell2744-castellano2-v4,17724,10.1448,jw08051-o009_t004_miri_p750l,data/miri_spectra_csv/abell2744-castellano2-v4_17724_miri_x1d.csv,387,ok
